(course-core-03)=

# Module 3: The Multi-Source Assembler

**Welcome back, Apprentice Master.** In Module 1 and Module 2, you learned that MolSysMT handles individual files, objects, and native structures as distinct **Forms**.

However, real computational workflows often split data across multiple files. For example, a heavy molecular dynamics simulation might store its structural topology in a PDB or H5MSM file, while storing thousands of coordinate frames in a binary DCD or XTC trajectory file.

Traditionally, loading a topology from one file and a trajectory from another required complex merging scripts or third-party wrappers. In MolSysMT, you simply pass a Python list of complementary sources, and the framework virtually merges them into a single, unified molecular system.

```{admonition} Glossary: Combined Sources
:class: dropdown info
**Combined Sources** refers to MolSysMT's ability to treat a list of complementary data forms (e.g., a topology file and a trajectory file) as a single, virtual molecular system without requiring manual array merging or file duplication.
```

(course-core-03-learning-outcomes)=
> **🎯 Learning Outcomes**
>
> By the end of this module, you will be able to:
> - Combine separate topology and trajectory files into a single virtual system list.
> - Validate composite systems using `msm.is_a_molecular_system()`.
> - Trace attribute origins and precedence rules using `msm.where_is_attribute()`.
> - Extract data arrays seamlessly using `msm.get()` while respecting coordinate array dimensions `(n_structures, n_atoms, 3)`.

### 1. The Virtual Multi-Source System

Let's begin by loading the topology and trajectory files of our solvated **Chicken Villin HP35** demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load topology file (H5MSM) and trajectory file (DCD)
topology_file = systems['chicken villin HP35']['chicken_villin_HP35_solvated.h5msm']
trajectory_file = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']

# Combine them into a single multi-source list
composite_system = [topology_file, trajectory_file]

print(f"Is it a valid system? {msm.is_a_molecular_system(composite_system)}")
print(f"Canonical form: {msm.get_form(composite_system)}")

Is it a valid system? True
Canonical form: ['file:h5msm', 'file:dcd']


:::{hint}
:class: dropdown
**msm.is_a_molecular_system()**: Validates whether a single item or list of items forms a valid, complementary molecular system. See API doc: {func}`molsysmt.basic.is_a_molecular_system`.
:::

### 2. Attribute Precedence: `msm.where_is_attribute()`

When a multi-source list contains complementary items, MolSysMT routes queries to whichever item provides the required data. 

If two items in your list provide the same attribute (for instance, if both the H5MSM topology and DCD trajectory contain coordinates), MolSysMT follows a simple **Rule of Precedence**: the **first** item in the list providing that attribute takes priority.

We can inspect which file provides a specific attribute using **`msm.where_is_attribute()`**:

In [2]:
# Trace the origin of 'atom_name'
source_names = msm.where_is_attribute(composite_system, 'atom_name')
print(f"Atom names are coming from: {source_names}")

# Trace the origin of 'coordinates'
source_coords = msm.where_is_attribute(composite_system, 'coordinates')
print(f"Coordinates are coming from: {source_coords}")

Atom names are coming from: ('/home/diego/repos@uibcdf/molsysmt/molsysmt/data/h5msm/chicken_villin_HP35_solvated.h5msm', 'file:h5msm')


Coordinates are coming from: ('/home/diego/repos@uibcdf/molsysmt/molsysmt/data/dcd/traj_chicken_villin_HP35_solvated.dcd', 'file:dcd')


:::{hint}
:class: dropdown
**msm.where_is_attribute()**: Identifies which item in a multi-source list provides a specific attribute. See API doc: {func}`molsysmt.basic.where_is_attribute`.
:::

### 3. Transparent Extraction and Coordinate Geometry: `msm.get()`

Because MolSysMT abstracts multi-source lists as unified systems, you can extract topological and structural attributes simultaneously using **`msm.get()`**.

One vital invariant to keep in mind: in MolSysMT, spatial coordinates are always returned as a **3D NumPy array** with the shape:  
`(n_structures, n_atoms, 3)` (where the last dimension represents X, Y, Z coordinates).

This 3D shape is strictly preserved across all single-frame or multi-frame systems to ensure downstream analytical scripts remain fully consistent.

In [3]:
# Extract atom names and coordinates for the first 5 atoms across the trajectory
names, coords = msm.get(composite_system, selection=[0, 1, 2, 3, 4], atom_name=True, coordinates=True)

print(f"Atom names: {names}")
print(f"Coordinates array shape: {coords.shape} (n_structures, n_atoms, spatial:x,y,z)")

# Y coordinate (index 1) of the 3rd atom (index 2) in the 4th structure (index 3)
print(f"Y coordinate of 3rd atom in 4th frame: {coords[3, 2, 1]}")

Atom names: ['C', 'O', 'CH3', 'H1', 'H2']
Coordinates array shape: (20, 5, 3) (n_structures, n_atoms, spatial:x,y,z)
Y coordinate of 3rd atom in 4th frame: 1.0159252166748045 nanometer


:::{hint}
:class: dropdown
**msm.get()**: Queries topological, structural, or physical attributes from a molecular system using selection queries. See API doc: {func}`molsysmt.basic.get`.
:::

--- 

### 🏆 Challenge 3: The System Weaver

1. Load a PDB ID (`'pdb_id:181L'`) and instantiate a native object (`msm.native.MolSys()`).
2. Pass them together in a list: `[msm.native.MolSys(), 'pdb_id:181L']`.
3. Use `msm.is_a_molecular_system()` to verify if the list is recognized as a valid molecular system.
4. Use `msm.where_is_attribute()` to check which item provides the atom names.

Mastering multi-source systems allows you to combine static structural PDB files with heavy trajectory files seamlessly. In [Module 4: Visualizing Anything](../00_Common_Core/04_Visualizing_Anything.ipynb), we will learn how to render interactive 3D visualizations of any system.

```{key-takeaway}
MolSysMT allows you to virtually merge multiple data sources into a single system by simply passing them as a list, with the first source taking precedence for overlapping attributes.
```

(course-core-03-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.is_a_molecular_system` — Molecular system validation engine.
- {func}`molsysmt.basic.where_is_attribute` — Multi-source attribute origin inspector.
- {func}`molsysmt.basic.get` — Unified attribute extraction engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 2: Native Forms](../00_Common_Core/02_Native_Forms.ipynb)
- Next Module: [Module 4: Visualizing Anything](../00_Common_Core/04_Visualizing_Anything.ipynb)
- User Guide: {ref}`user-foundations`
:::